In [1]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')
 
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_curve, auc
)

In [2]:
#Set the emojis 

POSITIVE_EMOJIS  = set('🥳🤩🔥😁😍✨💯🎉👍🌟😊')
NEUTRAL_EMOJIS   = set('🫤🙂🤷😐')
NEGATIVE_EMOJIS  = set('😒😠😞💔👎😤😢')
SARCASTIC_EMOJIS = set('🙄👏😂🙃')
 
SOCIAL_SLANG = {
    'tbh': 'to be honest', 'ngl': 'not gonna lie', 'omg': 'oh my god',
    'idk': 'i do not know', 'fr':  'for real',      'highkey': 'really',
    'lowkey': 'slightly',   'lit': 'amazing',        'goat': 'best ever',
    'sus':  'suspicious',   'slay': 'excellent',     'based': 'good',
    'cap':  'lie',          'no cap': 'truth',       'bussin': 'great food',
    'vibe': 'feeling',
}

In [3]:
#Emoji processor
def emoji_to_tag(text):
    tags = []
    for char in str(text):
        if char in POSITIVE_EMOJIS:   tags.append('EMOJI_POS')
        elif char in NEGATIVE_EMOJIS: tags.append('EMOJI_NEG')
        elif char in NEUTRAL_EMOJIS:  tags.append('EMOJI_NEU')
        elif char in SARCASTIC_EMOJIS:tags.append('EMOJI_SARC')
    clean = re.sub(r'[\U0001F300-\U0001FFFF\U00002702-\U000027B0]+', '', str(text))
    return clean.strip() + (' ' + ' '.join(tags) if tags else '')

In [4]:
#Slang processor
def expand_slang(text):
    words = text.split()
    out, i = [], 0
    while i < len(words):
        w = words[i].lower()
        out.append(SOCIAL_SLANG.get(w, words[i]))
        i += 1
    return ' '.join(out)
 
def normalize_elongation(text):
    return re.sub(r'(.)\1{2,}', r'\1', text)

In [5]:
#Text processor
def expand_slang(text):
    words = text.split()
    out, i = [], 0
    while i < len(words):
        w = words[i].lower()
        out.append(SOCIAL_SLANG.get(w, words[i]))
        i += 1
    return ' '.join(out)
 
def normalize_elongation(text):
    return re.sub(r'(.)\1{2,}', r'\1', text)

In [6]:
def preprocess_sentiment(text):
    """For social media text — handles emojis, slang, elongation."""
    text = emoji_to_tag(text)
    text = expand_slang(text)
    text = normalize_elongation(text)
    text = text.lower()
    text = re.sub(r'[^a-z\s_]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()
 
def preprocess_sarcasm(text):
    """For news headlines — simpler, no emojis."""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

In [7]:
# ──────────────────────────────────────────────────────────────────────────────
# LOAD DATA
# ──────────────────────────────────────────────────────────────────────────────
 
print("=" * 60)
print("  Loading datasets...")
print("=" * 60)
 
# ── Sarcasm: combine + deduplicate ──────────────────────────────────────────
df1 = pd.read_json('Sarcasm_Headlines_Dataset.json', lines=True)
df2 = pd.read_json('Sarcasm_Headlines_Dataset_v2.json', lines=True)
sarc_df = (pd.concat([df1, df2], ignore_index=True)
             .drop_duplicates(subset=['headline'])
             .reset_index(drop=True))

  Loading datasets...


In [8]:
print(f"\nSarcasm dataset : {len(df1):,} + {len(df2):,} → {len(sarc_df):,} after dedup")
print(f"  Sarcastic     : {(sarc_df['is_sarcastic']==1).sum():,}")
print(f"  Not Sarcastic : {(sarc_df['is_sarcastic']==0).sum():,}")


Sarcasm dataset : 26,709 + 28,619 → 28,503 after dedup
  Sarcastic     : 13,552
  Not Sarcastic : 14,951


In [9]:
# ── Sentiment: load, drop Sarcastic rows ────────────────────────────────────
sent_train = pd.read_csv('social_media_sentiment_train.csv')
sent_test  = pd.read_csv('social_media_sentiment_test.csv')
 
# Remove rows labelled Sarcastic — sentiment model only handles Pos/Neg/Neu
sent_train = sent_train[sent_train['label'] != 'Sarcastic'].reset_index(drop=True)
sent_test  = sent_test[sent_test['label']  != 'Sarcastic'].reset_index(drop=True)
 
print(f"\nSentiment train : {len(sent_train):,} rows")
print(f"Sentiment test  : {len(sent_test):,} rows")
print(f"  Labels: {sent_train['label'].value_counts().to_dict()}")


Sentiment train : 7,298 rows
Sentiment test  : 1,480 rows
  Labels: {'Positive': 2930, 'Negative': 2850, 'Neutral': 1518}


In [10]:
print("\n" + "=" * 60)
print("  STAGE 1: Training Sarcasm Detector")
print("=" * 60)
 
sarc_df['clean'] = sarc_df['headline'].apply(preprocess_sarcasm)
 
X_sarc = sarc_df['clean']
y_sarc = sarc_df['is_sarcastic']
 
X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(
    X_sarc, y_sarc, test_size=0.2, random_state=42, stratify=y_sarc
)


  STAGE 1: Training Sarcasm Detector


In [11]:
sarcasm_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 3),
        max_features=60000,
        sublinear_tf=True,
        min_df=2,
    )),
    ('clf', ComplementNB(alpha=0.1))
])

In [12]:
sarcasm_model.fit(X_s_train, y_s_train)
sarc_preds = sarcasm_model.predict(X_s_test)
sarc_probs = sarcasm_model.predict_proba(X_s_test)[:, 1]

In [13]:
sarc_acc = accuracy_score(y_s_test, sarc_preds)
sarc_f1  = f1_score(y_s_test, sarc_preds)
sarc_cv  = cross_val_score(sarcasm_model, X_sarc, y_sarc, cv=5, scoring='f1').mean()
 
print(f"\n  Accuracy : {sarc_acc:.4f}")
print(f"  F1 Score : {sarc_f1:.4f}")
print(f"  CV F1    : {sarc_cv:.4f}")
print("\n", classification_report(y_s_test, sarc_preds,
      target_names=['Not Sarcastic', 'Sarcastic']))


  Accuracy : 0.8358
  F1 Score : 0.8316
  CV F1    : 0.8277

                precision    recall  f1-score   support

Not Sarcastic       0.86      0.82      0.84      2990
    Sarcastic       0.81      0.85      0.83      2711

     accuracy                           0.84      5701
    macro avg       0.84      0.84      0.84      5701
 weighted avg       0.84      0.84      0.84      5701



In [14]:
print("=" * 60)
print("  STAGE 2: Training Sentiment Model")
print("=" * 60)
 
sent_train['clean'] = sent_train['text'].apply(preprocess_sentiment)
sent_test['clean']  = sent_test['text'].apply(preprocess_sentiment)
 
sentiment_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 3),
        max_features=50000,
        sublinear_tf=True,
        min_df=2,
    )),
    ('clf', ComplementNB(alpha=0.1))
])

  STAGE 2: Training Sentiment Model


In [15]:
sentiment_model.fit(sent_train['clean'], sent_train['label'])
sent_preds = sentiment_model.predict(sent_test['clean'])
 
sent_acc = accuracy_score(sent_test['label'], sent_preds)
sent_f1  = f1_score(sent_test['label'], sent_preds, average='weighted')
sent_cv  = cross_val_score(sentiment_model, sent_train['clean'],
                           sent_train['label'], cv=5, scoring='f1_weighted').mean()
 
print(f"\n  Accuracy : {sent_acc:.4f}")
print(f"  F1 Score : {sent_f1:.4f}")
print(f"  CV F1    : {sent_cv:.4f}")
print("\n", classification_report(sent_test['label'], sent_preds))


  Accuracy : 1.0000
  F1 Score : 1.0000
  CV F1    : 0.9992

               precision    recall  f1-score   support

    Negative       1.00      1.00      1.00       543
     Neutral       1.00      1.00      1.00       402
    Positive       1.00      1.00      1.00       535

    accuracy                           1.00      1480
   macro avg       1.00      1.00      1.00      1480
weighted avg       1.00      1.00      1.00      1480



In [16]:
FLIP_MAP = {'Positive': 'Negative', 'Negative': 'Positive', 'Neutral': 'Neutral'}
 
def predict_pipeline(texts, sarcasm_threshold=0.5):
    cleaned_sarc = [preprocess_sarcasm(t) for t in texts]
    cleaned_sent = [preprocess_sentiment(t) for t in texts]
 
    sarc_probs  = sarcasm_model.predict_proba(cleaned_sarc)[:, 1]
    sent_labels = sentiment_model.predict(cleaned_sent)
    sent_probs  = sentiment_model.predict_proba(cleaned_sent)
 
    rows = []
    for text, sp, sl, sprob in zip(texts, sarc_probs, sent_labels, sent_probs):
        is_sarc = sp >= sarcasm_threshold
        final   = FLIP_MAP[sl] if is_sarc else sl
        rows.append({
            'text':            text,
            'raw_sentiment':   sl,
            'is_sarcastic':    is_sarc,
            'sarcasm_prob':    round(float(sp), 3),
            'final_sentiment': final,
            'confidence':      round(float(max(sprob)), 3),
        })
    return pd.DataFrame(rows)

In [17]:
print("\n" + "=" * 60)
print("  LIVE INFERENCE DEMO")
print("=" * 60)
 
demo = [
    "This is absolutely amazing!",
    "oh wow, totally helpful 🙄",
    "the product was okay I guess",
    "worst experience ever, thanks a lot",
    "I genuinely love this service 😍",
    "area man proudly declares he has never read a book",        # sarcastic headline style
    "scientists discover new species of bird in amazon",         # neutral headline style
]
 


  LIVE INFERENCE DEMO


In [18]:
results = predict_pipeline(demo)
pd.set_option('display.max_colwidth', 45)
print(results[['text','raw_sentiment','is_sarcastic','sarcasm_prob','final_sentiment']].to_string(index=False))

                                              text raw_sentiment  is_sarcastic  sarcasm_prob final_sentiment
                       This is absolutely amazing!      Positive         False         0.076        Positive
                         oh wow, totally helpful 🙄      Negative         False         0.472        Negative
                      the product was okay I guess       Neutral          True         0.868         Neutral
               worst experience ever, thanks a lot      Negative         False         0.360        Negative
                   I genuinely love this service 😍      Positive         False         0.104        Positive
area man proudly declares he has never read a book      Negative          True         0.988        Positive
 scientists discover new species of bird in amazon      Negative          True         0.997        Positive


In [19]:
BG      = '#0f172a'
PANEL   = '#1e293b'
WHITE   = 'white'
MUTED   = '#94a3b8'
COLORS  = {'Positive':'#22c55e','Negative':'#ef4444','Neutral':'#64748b'}
C_SARC  = ['#60a5fa','#f472b6']   # not-sarcastic, sarcastic
 
fig = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor(BG)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.38)
 
tk = dict(colors=MUTED, labelsize=9)
lk = dict(color=MUTED, fontsize=10)
titk = dict(color=WHITE, fontsize=12, fontweight='bold', pad=10)

<Figure size 2000x1600 with 0 Axes>

In [20]:
ax = fig.add_subplot(gs[0, 0])
ax.set_facecolor(PANEL)
counts = sarc_df['is_sarcastic'].value_counts().sort_index()
ax.bar(['Not Sarcastic','Sarcastic'], counts.values, color=C_SARC, edgecolor='none', width=0.5)
for i, v in enumerate(counts.values):
    ax.text(i, v + 100, f'{v:,}', ha='center', color=WHITE, fontsize=10)
ax.set_title('Sarcasm Dataset Balance', **titk)
ax.tick_params(**tk); ax.spines[:].set_visible(False)
ax.set_ylabel('Count', **lk)

Text(0, 0.5, 'Count')

In [21]:
ax = fig.add_subplot(gs[0, 1])
ax.set_facecolor(PANEL)
order  = ['Positive','Negative','Neutral']
scounts = sent_train['label'].value_counts().reindex(order)
ax.bar(order, scounts.values, color=[COLORS[c] for c in order], edgecolor='none', width=0.5)
for i, v in enumerate(scounts.values):
    ax.text(i, v + 20, str(v), ha='center', color=WHITE, fontsize=10)
ax.set_title('Sentiment Dataset Balance', **titk)
ax.tick_params(**tk); ax.spines[:].set_visible(False)
ax.set_ylabel('Count', **lk)

Text(0, 0.5, 'Count')

In [22]:
ax = fig.add_subplot(gs[0, 2])
ax.set_facecolor(PANEL)
metrics = ['Accuracy', 'F1', 'CV F1']
sarc_vals = [sarc_acc, sarc_f1, sarc_cv]
sent_vals = [sent_acc, sent_f1, sent_cv]
x = np.arange(3); w = 0.3
ax.bar(x - w/2, sarc_vals, w, label='Stage 1 (Sarcasm)', color='#818cf8', edgecolor='none')
ax.bar(x + w/2, sent_vals, w, label='Stage 2 (Sentiment)', color='#34d399', edgecolor='none')
ax.set_xticks(x); ax.set_xticklabels(metrics, color=MUTED, fontsize=9)
ax.set_ylim(0, 1.15)
for i, (sv, stv) in enumerate(zip(sarc_vals, sent_vals)):
    ax.text(i - w/2, sv + 0.02, f'{sv:.2f}', ha='center', color=WHITE, fontsize=8)
    ax.text(i + w/2, stv + 0.02, f'{stv:.2f}', ha='center', color=WHITE, fontsize=8)
ax.set_title('Stage 1 vs Stage 2 Metrics', **titk)
ax.legend(fontsize=8, facecolor=BG, labelcolor=WHITE)
ax.tick_params(**tk); ax.spines[:].set_visible(False)

In [23]:
ax = fig.add_subplot(gs[1, 0])
ax.set_facecolor(PANEL)
cm = confusion_matrix(y_s_test, sarc_preds)
cm_n = cm / cm.sum(axis=1, keepdims=True)
im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1, aspect='auto')
labels = ['Not Sarc', 'Sarcastic']
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(labels, color=MUTED, fontsize=9)
ax.set_yticklabels(labels, color=MUTED, fontsize=9)
ax.set_xlabel('Predicted', **lk); ax.set_ylabel('Actual', **lk)
ax.set_title('Sarcasm Confusion Matrix', **titk)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm_n[i,j]:.2f}\n({cm[i,j]:,})',
                ha='center', va='center', fontsize=10,
                color=WHITE if cm_n[i,j] > 0.5 else PANEL)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

In [24]:
ax = fig.add_subplot(gs[1, 1])
ax.set_facecolor(PANEL)
order3 = ['Positive','Negative','Neutral']
cm2  = confusion_matrix(sent_test['label'], sent_preds, labels=order3)
cm2n = cm2 / cm2.sum(axis=1, keepdims=True)
im2  = ax.imshow(cm2n, cmap='Greens', vmin=0, vmax=1, aspect='auto')
ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
ax.set_xticklabels(order3, color=MUTED, fontsize=9)
ax.set_yticklabels(order3, color=MUTED, fontsize=9)
ax.set_xlabel('Predicted', **lk); ax.set_ylabel('Actual', **lk)
ax.set_title('Sentiment Confusion Matrix', **titk)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cm2n[i,j]:.2f}\n({cm2[i,j]})',
                ha='center', va='center', fontsize=9,
                color=WHITE if cm2n[i,j] > 0.5 else PANEL)
fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)

In [25]:
ax = fig.add_subplot(gs[1, 2])
ax.set_facecolor(PANEL)
fpr, tpr, _ = roc_curve(y_s_test, sarc_probs)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color='#818cf8', lw=2, label=f'AUC = {roc_auc:.3f}')
ax.plot([0,1],[0,1], color=MUTED, lw=1, linestyle='--')
ax.set_xlabel('False Positive Rate', **lk)
ax.set_ylabel('True Positive Rate', **lk)
ax.set_title('ROC Curve — Sarcasm Detector', **titk)
ax.legend(fontsize=9, facecolor=BG, labelcolor=WHITE)
ax.tick_params(**tk); ax.spines[:].set_visible(False)

In [26]:
ax = fig.add_subplot(gs[2, 0])
ax.set_facecolor(PANEL)
feat_names = sarcasm_model.named_steps['tfidf'].get_feature_names_out()
log_probs  = sarcasm_model.named_steps['clf'].feature_log_prob_
top_sarc   = np.argsort(log_probs[1])[-12:][::-1]
top_nsarc  = np.argsort(log_probs[0])[-12:][::-1]
ax.barh(range(12), log_probs[1][top_sarc][::-1] - log_probs[0][top_sarc][::-1],
        color='#f472b6', edgecolor='none')
ax.set_yticks(range(12))
ax.set_yticklabels(feat_names[top_sarc][::-1], fontsize=8, color=WHITE)
ax.set_title('Top Sarcasm Signal Words', **titk)
ax.tick_params(axis='x', **tk); ax.spines[:].set_visible(False)
ax.axvline(0, color=MUTED, lw=0.8)

In [27]:
sent_feat  = sentiment_model.named_steps['tfidf'].get_feature_names_out()
sent_lp    = sentiment_model.named_steps['clf'].feature_log_prob_
sent_cls   = list(sentiment_model.classes_)
 
for col, cls in enumerate(order3):
    ax = fig.add_subplot(gs[2, col+1] if col < 2 else gs[2, 2])
    if col >= 2:
        break
    ax.set_facecolor(PANEL)
    idx     = sent_cls.index(cls)
    top_idx = np.argsort(sent_lp[idx])[-10:][::-1]
    scores  = sent_lp[idx][top_idx]
    norm    = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    ax.barh(range(10), norm[::-1], color=COLORS[cls], edgecolor='none')
    ax.set_yticks(range(10))
    ax.set_yticklabels(sent_feat[top_idx][::-1], fontsize=8, color=WHITE)
    ax.set_title(f'Top {cls} Features', **titk)
    ax.tick_params(axis='x', **tk); ax.spines[:].set_visible(False)

In [28]:
ax = fig.add_subplot(gs[2, 2])
ax.set_facecolor(PANEL)
idx     = sent_cls.index('Neutral')
top_idx = np.argsort(sent_lp[idx])[-10:][::-1]
scores  = sent_lp[idx][top_idx]
norm    = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
ax.barh(range(10), norm[::-1], color=COLORS['Neutral'], edgecolor='none')
ax.set_yticks(range(10))
ax.set_yticklabels(sent_feat[top_idx][::-1], fontsize=8, color=WHITE)
ax.set_title('Top Neutral Features', **titk)
ax.tick_params(axis='x', **tk); ax.spines[:].set_visible(False)
 
fig.suptitle(
    f'Two-Stage Pipeline  |  Sarcasm F1: {sarc_f1:.3f}  |  Sentiment Acc: {sent_acc*100:.1f}%',
    color=WHITE, fontsize=15, fontweight='bold', y=0.98
)

Text(0.5, 0.98, 'Two-Stage Pipeline  |  Sarcasm F1: 0.832  |  Sentiment Acc: 100.0%')

In [30]:
plt.savefig('two_stage_pipeline_results.png',
            dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print("\nPlot saved.")


Plot saved.
